In [1]:
!pip install -q opencv-python-headless
!git clone https://github.com/ultralytics/yolov5
!pip install -q -r yolov5/requirements.txt


Cloning into 'yolov5'...
remote: Enumerating objects: 18366, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 18366 (delta 35), reused 18 (delta 18), pack-reused 18305 (from 2)
Receiving objects: 100% (18366/18366), 17.48 MiB | 25.67 MiB/s, done.
Resolving deltas: 100% (12479/12479), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.0 MB/s eta 0:00:00


In [2]:
from google.colab import files
files.upload()

KeyboardInterrupt: 

In [ ]:
import cv2
import numpy as np
import torch
import time
from collections import defaultdict


#CONFIGURATION


VIDEO_SOURCE = "Dashcam.mp4"
OUTPUT_VIDEO = "output_adas.mp4"

YOLO_VARIANT = "yolov5l"
DISPLAY_ZOOM = 1.45

LANE_WARNING_THRESHOLD = 75
COLLISION_BOX_AREA_THRESHOLD = 21000
MIN_CONFIDENCE = 0.42

# Visual style
COLOR_LANE = (200, 220, 0)
COLOR_ROI = (0, 240, 240)
COLOR_SAFE = (60, 255, 80)
COLOR_ALERT = (0, 40, 255)
COLOR_TEXT = (245, 245, 245)
COLOR_PANEL_BG = (0, 0, 0)


VEHICLE_CATEGORIES = {"car", "truck", "bus", "motorcycle"}

# TRACKING & COUNTING


next_vehicle_id = 1
vehicle_centers = {}               # id → (cx, cy)
counted_vehicle_ids = set()

total_vehicle_count = 0
vehicle_class_count = defaultdict(int)



In [ ]:
print("Loading YOLOv5 model...")
detector = torch.hub.load('ultralytics/yolov5', YOLO_VARIANT, pretrained=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
detector.to(device)

if device == 'cuda':
    detector.half()

detector.eval()
print("Model loaded on:", device)


In [ ]:
cap = cv2.VideoCapture(VIDEO_SOURCE)
fps = cap.get(cv2.CAP_PROP_FPS)

ret, frame = cap.read()
assert ret, "Video not found or unreadable"

TARGET_W = 704
scale = TARGET_W / frame.shape[1]
frame = cv2.resize(frame, (TARGET_W, int(frame.shape[0] * scale)))
h, w = frame.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    fps,
    (int(w * DISPLAY_ZOOM), int(h * DISPLAY_ZOOM))
)

print("Video writer initialized")


In [ ]:
def create_bottom_trapezoid_mask(h, w):
    mask = np.zeros((h, w), dtype=np.uint8)
    pts = np.array([
        [w*0.08, h],
        [w*0.42, h*0.62],
        [w*0.58, h*0.62],
        [w*0.92, h]
    ], np.int32)
    cv2.fillPoly(mask, [pts], 255)
    return mask


def average_lines_to_lanes(lines, h):
    if lines is None:
        return None, None

    left, right = [], []
    for l in lines:
        x1,y1,x2,y2 = l[0]
        if x2 == x1:
            continue
        m = (y2-y1)/(x2-x1)
        b = y1 - m*x1
        if m < -0.4:
            left.append((m,b))
        elif m > 0.4:
            right.append((m,b))

    def build(params):
        if not params:
            return None
        m,b = np.mean(params, axis=0)
        y1,y2 = h, int(h*0.58)
        x1 = int((y1-b)/m)
        x2 = int((y2-b)/m)
        return (x1,y1,x2,y2)

    return build(left), build(right)


def is_between_lanes(x, y, L, R):
    if not (L and R):
        return False
    lx1,ly1,lx2,ly2 = L
    rx1,ry1,rx2,ry2 = R
    ml = (ly2-ly1)/(lx2-lx1)
    bl = ly1 - ml*lx1
    mr = (ry2-ry1)/(rx2-rx1)
    br = ry1 - mr*rx1
    xl = (y-bl)/ml
    xr = (y-br)/mr
    return xl < x < xr


def forward_roi(w, h):
    return np.array([
        (int(w*0.28), int(h*0.48)),
        (int(w*0.72), int(h*0.48)),
        (int(w*0.96), h),
        (int(w*0.04), h)
    ], np.int32)


In [ ]:
frame_id = 0
last_left, last_right = None, None

def assign_vehicle_id(cx, cy):
    global next_vehicle_id
    for vid, (px, py) in vehicle_centers.items():
        if abs(cx - px) < 70 and abs(cy - py) < 70:
            return vid
    vid = next_vehicle_id
    next_vehicle_id += 1
    return vid


while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    frame = cv2.resize(frame, (TARGET_W, int(frame.shape[0] * scale)))
    h, w = frame.shape[:2]

    #Lane detection (every 4 frames)
    if frame_id % 4 == 0:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 60, 160)
        edges = cv2.bitwise_and(edges, create_bottom_trapezoid_mask(h, w))
        lines = cv2.HoughLinesP(
            edges, 1, np.pi / 180, 60,
            minLineLength=90, maxLineGap=45
        )
        last_left, last_right = average_lines_to_lanes(lines, h)

    left_lane, right_lane = last_left, last_right

    if left_lane:
        cv2.line(frame, left_lane[:2], left_lane[2:], COLOR_LANE, 4)
    if right_lane:
        cv2.line(frame, right_lane[:2], right_lane[2:], COLOR_LANE, 4)

    # Forward ROI
    roi = forward_roi(w, h)
    cv2.polylines(frame, [roi], True, COLOR_ROI, 2)

    #TOTAL VEHICLE COUNT (TEXT ONLY)
    cv2.putText(
        frame,
        f"TOTAL VEHICLES: {total_vehicle_count}",
        (20, 40),
        cv2.FONT_HERSHEY_DUPLEX,
        0.8,
        COLOR_TEXT,
        2
    )

    #Object Detection
    results = detector(frame, size=640)
    detections = results.xyxy[0].cpu().numpy()

    danger = False
    primary = None

    for x1, y1, x2, y2, conf, cid in detections:
        label = detector.names[int(cid)]

        if label not in VEHICLE_CATEGORIES or conf < MIN_CONFIDENCE:
            continue

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        area = (x2 - x1) * (y2 - y1)

        #ID assignment
        vid = assign_vehicle_id(cx, cy)
        vehicle_centers[vid] = (cx, cy)

        #Count once when entering ROI
        in_roi = cv2.pointPolygonTest(roi, (cx, cy), False) >= 0
        if in_roi and vid not in counted_vehicle_ids:
            counted_vehicle_ids.add(vid)
            total_vehicle_count += 1

        #Danger evaluation
        in_lane = is_between_lanes(cx, cy, left_lane, right_lane)
        color = COLOR_SAFE

        if in_roi and in_lane and (area > COLLISION_BOX_AREA_THRESHOLD or y2 > h * 0.84):
            color = COLOR_ALERT
            danger = True
            if not primary or cy > primary[3]:
                primary = (int(x1), int(y1), int(x2), int(y2))

        #Draw bounding box + vehicle type
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(
            frame,
            label.upper(),
            (int(x1), int(y1) - 7),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            color,
            2
        )

    #Highlight primary threat
    if primary:
        cv2.rectangle(frame, primary[:2], primary[2:], COLOR_ALERT, 5)
        cv2.putText(
            frame,
            "!! THREAT !!",
            (primary[0], primary[1] - 10),
            cv2.FONT_HERSHEY_DUPLEX,
            0.7,
            COLOR_ALERT,
            2
        )

    #Collision warning banner
    if danger:
        cv2.rectangle(
            frame,
            (w // 2 - 320, 15),
            (w // 2 + 320, 75),
            COLOR_ALERT,
            -1
        )
        cv2.putText(
            frame,
            "COLLISION IMMINENT",
            (w // 2 - 280, 60),
            cv2.FONT_HERSHEY_DUPLEX,
            1.1,
            COLOR_TEXT,
            3
        )

    #Write frame
    shown = cv2.resize(frame, None, fx=DISPLAY_ZOOM, fy=DISPLAY_ZOOM)
    out.write(shown)

print("Processing finished")




In [ ]:
cap.release()
out.release()
print("Video saved:", OUTPUT_VIDEO)


In [ ]:
from IPython.display import HTML

HTML(f"""
<video width="900" controls>
  <source src="{OUTPUT_VIDEO}" type="video/mp4">
</video>
""")
